# 04 · Neural network on the NNPA — SepsisWatch Z

1. Loads `features.pkl` (140 features) with the same split as `03`: **train/val = hospital A**, **test = hospital B**.
2. Trains a PyTorch MLP on CPU with early stopping on validation AUPRC.
3. Runs inference on the **Telum II on-chip AI accelerator (NNPA)** and checks it matches CPU quality.
4. Saves **predictions** (for the ensemble and `05_eval`) and **artifacts** for the `scorer`.
5. Measures **CPU vs. NNPA latency** with the real model (p50/p95) → evidence `11` and `12`.

> If the kernel already imported `torch`, use **Kernel → Restart Kernel** and run everything from the top.

## 1. Environment

In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"   # before importing torch (IBM Churn lab pattern)

import json, time, copy, platform, warnings, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

try:
    import torch_nnpa
    NNPA_OK = torch.nnpa.is_available()
except Exception as e:
    print("Could not load torch_nnpa:", e)
    NNPA_OK = False

print(f"Machine: {platform.machine()} | torch {torch.__version__} | CPU threads: {torch.get_num_threads()}")
print(f"NNPA available: {NNPA_OK}" + (f" | devices: {torch.nnpa.device_count()}" if NNPA_OK else ""))

## 2. Configuration

In [ ]:
DATA_DIR    = os.path.expanduser("~/shared/sepsis/data/processed")
RAW_ALL     = f"{DATA_DIR}/sepsis_all.pkl"
FEAT_PATH   = f"{DATA_DIR}/features.pkl"
FLIST_PATH  = f"{DATA_DIR}/feature_list.json"
MODEL_DIR   = os.path.expanduser("~/shared/sepsis/models")    # outside the repo
REPO        = os.path.abspath("..")                            # notebooks run from notebooks/
FIG_DIR     = f"{REPO}/docs/figures"

LABEL_COL   = "SepsisLabel"
DROP_FEATS  = []          # keep identical in 03 and 04, e.g. ["ICULOS"]
SEED        = 42

HIDDEN      = [256, 128, 64]
DROPOUT     = 0.2
EPOCHS      = 12
PATIENCE    = 3           # epochs without val AUPRC improvement before stopping
TRAIN_BATCH = 2048
LR          = 1e-3

BATCH_SIZES = [1, 12, 64, 256, 1024, 4096]   # 1 = one bed, 12 = one ICU ward
WARMUP, ITERS = 20, 200

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)

## 3. Load features and split (identical to `03`)

In [ ]:
t0 = time.perf_counter()
df = pd.read_pickle(FEAT_PATH)
with open(FLIST_PATH) as f:
    flist = json.load(f)
FEATURES = flist if isinstance(flist, list) else (flist.get("features") or flist.get("feature_list") or next(iter(flist.values())))
FEATURES = [c for c in FEATURES if c not in DROP_FEATS]
missing = [c for c in FEATURES + [LABEL_COL, "patient_id", "hour"] if c not in df.columns]
assert not missing, f"Columns not found: {missing[:10]}"
print(f"features.pkl: {df.shape} loaded in {time.perf_counter()-t0:.1f}s | {df.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"Features used: {len(FEATURES)} | other columns: {[c for c in df.columns if c not in FEATURES][:12]}")

SPLIT_COL = next((c for c in ("split", "set", "fold", "subset") if c in df.columns), None)
if SPLIT_COL:
    split = df[SPLIT_COL].astype(str).str.lower().replace({"validation": "val", "valid": "val"})
    print(f"Using the existing split (column '{SPLIT_COL}')")
else:
    warnings.warn("No split column found: rebuilding it (same code in notebooks 03 and 04).")
    split = pd.Series("test", index=df.index)
    a = df["hospital"].astype(str) == "A"
    per_patient = df.loc[a].groupby("patient_id")[LABEL_COL].max()
    rng = np.random.default_rng(SEED)
    val_ids = set()
    for lab in (0, 1):
        ids = per_patient.index[per_patient.values == lab].to_numpy()
        rng.shuffle(ids)
        val_ids.update(ids[: int(round(len(ids) * 0.2))])
    split[a] = np.where(df.loc[a, "patient_id"].isin(val_ids), "val", "train")

idx = {s: np.flatnonzero(split.to_numpy() == s) for s in ("train", "val", "test")}
y = df[LABEL_COL].to_numpy().astype(np.float32)
for s, ii in idx.items():
    print(f"{s:>5}: {len(ii):>9,} rows | {df['patient_id'].iloc[ii].nunique():>6,} patients | label=1: {y[ii].mean():.2%}")
assert all(len(v) for v in idx.values()), "A split is empty"
assert not set(df["patient_id"].iloc[idx["train"]]) & set(df["patient_id"].iloc[idx["test"]]), "Patients shared by train and test"

## 4. Preprocessing (fitted on train only)
The network cannot take `NaN`: missing values are filled with the train median and standardized with the train mean and std. Everything stays `float32` and in place to protect the 8 GB of RAM.

In [ ]:
X = df[FEATURES].to_numpy(dtype=np.float32, copy=True)
META = df[[c for c in ("patient_id", "hospital", "hour") if c in df.columns]].copy()
META["patient_id"] = META["patient_id"].astype(str)
META["split"] = split.to_numpy()
META[LABEL_COL] = y
del df; gc.collect()

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    med = np.nanmedian(X[idx["train"]], axis=0)
med = np.where(np.isnan(med), 0, med).astype(np.float32)
for j in range(X.shape[1]):
    m = np.isnan(X[:, j])
    if m.any():
        X[m, j] = med[j]

mu = X[idx["train"]].mean(0, dtype=np.float64).astype(np.float32)
sd = X[idx["train"]].std(0, dtype=np.float64).astype(np.float32)
sd[sd == 0] = 1
X -= mu
X /= sd
print(f"X: {X.shape} {X.dtype} | {X.nbytes/1e9:.2f} GB | remaining NaN: {int(np.isnan(X).sum())}")

## 5. Train the MLP (CPU)

In [ ]:
def build_mlp(n_features, hidden, dropout):
    layers, prev = [], n_features
    for h in hidden:
        layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
        prev = h
    layers.append(nn.Linear(prev, 1))
    return nn.Sequential(*layers)

def infer(model, x_cpu, device):
    # end-to-end: to device -> forward -> sigmoid -> back to CPU (.cpu() waits for the chip)
    with torch.no_grad():
        return torch.sigmoid(model(x_cpu.to(device))).cpu()

def predict(model, rows, device, chunk=8192):
    return torch.cat([infer(model, Xall[rows[i:i+chunk]], device)
                      for i in range(0, len(rows), chunk)]).numpy().ravel()

Xall = torch.from_numpy(X)                        # shares memory with X (no copy)
yall = torch.from_numpy(y).unsqueeze(1)
tr_idx = torch.from_numpy(idx["train"])

model = build_mlp(X.shape[1], HIDDEN, DROPOUT)
n_pos = y[idx["train"]].sum(); n_neg = len(idx["train"]) - n_pos
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([n_neg / n_pos], dtype=torch.float32))
opt = torch.optim.Adam(model.parameters(), lr=LR)

best = {"auprc": -1, "epoch": 0, "state": None}
train_s = time.perf_counter()
for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.perf_counter(); total = 0.0
    perm = tr_idx[torch.randperm(len(tr_idx))]
    for i in range(0, len(perm), TRAIN_BATCH):
        b = perm[i:i + TRAIN_BATCH]
        opt.zero_grad()
        loss = loss_fn(model(Xall[b]), yall[b])
        loss.backward(); opt.step()
        total += loss.item() * len(b)
    model.eval()
    pv = predict(model, idx["val"], "cpu")
    auprc, auroc = average_precision_score(y[idx["val"]], pv), roc_auc_score(y[idx["val"]], pv)
    improved = auprc > best["auprc"]
    if improved:
        best = {"auprc": auprc, "epoch": ep, "state": copy.deepcopy(model.state_dict())}
    print(f"epoch {ep:>2} | loss {total/len(perm):.4f} | val AUROC {auroc:.4f} AUPRC {auprc:.4f} | {time.perf_counter()-t0:.1f}s {'*' if improved else ''}")
    if ep - best["epoch"] >= PATIENCE:
        print(f"Early stopping: no improvement in {PATIENCE} epochs")
        break
train_s = time.perf_counter() - train_s

model.load_state_dict(best["state"]); model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"\nBest epoch: {best['epoch']} (val AUPRC {best['auprc']:.4f}) | {n_params:,} parameters | {train_s:.0f}s")

## 6. NNPA inference and CPU vs. NNPA quality
The full validation set and **hospital B** are scored on both devices. If quality matches, the NNPA can be used in production with no loss of accuracy.

In [ ]:
model_nnpa = copy.deepcopy(model).to("nnpa").eval() if NNPA_OK else None
DEVICES = ["cpu"] + (["nnpa"] if NNPA_OK else [])

if NNPA_OK:   # evidence 11: the tensor lives on the NNPA
    x_demo = Xall[idx["test"][:4]].to("nnpa")
    with torch.no_grad():
        out_demo = torch.sigmoid(model_nnpa(x_demo))
    print("Evidence 11 → device:", out_demo.device, "| risk scores:", out_demo.cpu().numpy().ravel().round(4))

probs, score_s = {}, {}
for dev in DEVICES:
    m = model if dev == "cpu" else model_nnpa
    for s in ("val", "test"):
        t0 = time.perf_counter()
        probs[(dev, s)] = predict(m, idx[s], dev)
        score_s[(dev, s)] = time.perf_counter() - t0

rows = []
for s in ("val", "test"):
    for dev in DEVICES:
        p = probs[(dev, s)]
        rows.append({"split": s, "device": dev, "rows": len(p),
                     "AUROC": roc_auc_score(y[idx[s]], p), "AUPRC": average_precision_score(y[idx[s]], p),
                     "prevalence": y[idx[s]].mean(), "time_s": score_s[(dev, s)]})
quality = pd.DataFrame(rows)
display(quality.style.format({"AUROC": "{:.4f}", "AUPRC": "{:.4f}", "prevalence": "{:.4f}", "time_s": "{:.2f}", "rows": "{:,}"}))

if NNPA_OK:
    thr = np.quantile(probs[("cpu", "val")], 0.95)   # reference threshold: top 5% riskiest hours in val
    pc, pn = probs[("cpu", "test")], probs[("nnpa", "test")]
    print(f"Hospital B: mean abs difference {np.abs(pc-pn).mean():.6f}, max {np.abs(pc-pn).max():.6f} | "
          f"same decision at threshold {thr:.3f}: {((pc>=thr)==(pn>=thr)).mean():.4%}")
    print(f"Scoring all {len(pc):,} hospital B patient-hours: CPU {score_s[('cpu','test')]:.2f}s vs NNPA {score_s[('nnpa','test')]:.2f}s")

## 7. Save predictions (for the ensemble and `05_eval`) and artifacts (for the `scorer`)

In [ ]:
pred = META.iloc[np.concatenate([idx["val"], idx["test"]])].reset_index(drop=True)
for dev in DEVICES:
    pred[f"p_mlp_{dev}"] = np.concatenate([probs[(dev, "val")], probs[(dev, "test")]]).astype(np.float32)
pred.to_pickle(f"{MODEL_DIR}/preds_mlp.pkl")

torch.save(model.state_dict(), f"{MODEL_DIR}/mlp.pt")
np.savez(f"{MODEL_DIR}/mlp_scaler.npz", median=med, mean=mu, std=sd)
cfg = {"features": FEATURES, "hidden": HIDDEN, "dropout": DROPOUT, "label": LABEL_COL,
       "best_epoch": best["epoch"], "n_params": n_params,
       "preprocess": "x = nan->median; x = (x-mean)/std; p = sigmoid(model(x))",
       "metrics": quality.to_dict(orient="records")}
with open(f"{MODEL_DIR}/mlp_config.json", "w") as f:
    json.dump(cfg, f, indent=2, default=float)

print("Saved to", MODEL_DIR)
for fn in sorted(os.listdir(MODEL_DIR)):
    print(f"  {fn:<24} {os.path.getsize(f'{MODEL_DIR}/{fn}')/1e6:8.2f} MB")
print("preds_mlp.pkl columns:", list(pred.columns))

**For the `scorer`** — load the network on the NNPA and score one patient-hour:

```python
import json, numpy as np, torch, torch.nn as nn, torch_nnpa
cfg = json.load(open(".../models/mlp_config.json")); sc = np.load(".../models/mlp_scaler.npz")
def build_mlp(n, hidden, dropout):
    layers, prev = [], n
    for h in hidden: layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]; prev = h
    return nn.Sequential(*layers, nn.Linear(prev, 1))
model = build_mlp(len(cfg["features"]), cfg["hidden"], cfg["dropout"])
model.load_state_dict(torch.load(".../models/mlp.pt")); model.eval().to("nnpa")

def score(row):   # row: dict with the 140 features for that hour
    x = np.array([row.get(c, np.nan) for c in cfg["features"]], dtype=np.float32)
    x = np.where(np.isnan(x), sc["median"], x); x = (x - sc["mean"]) / sc["std"]
    with torch.no_grad():
        return float(torch.sigmoid(model(torch.from_numpy(x)[None].to("nnpa"))).cpu())
```

## 8. CPU vs. NNPA latency benchmark (real model) — evidence 12
End-to-end: includes the CPU↔NNPA copy and the `.cpu()` call inside the timer. Hospital B data.

In [ ]:
Xtest = X[idx["test"]]

def time_device(m, device, batch):
    n = len(Xtest)
    batches = [torch.from_numpy(Xtest[(i*batch) % max(1, n-batch+1):][:batch]) for i in range(min(ITERS, max(1, n // batch)))]
    for i in range(WARMUP):
        infer(m, batches[i % len(batches)], device)
    lat = np.empty(ITERS)
    for i in range(ITERS):
        xb = batches[i % len(batches)]
        t0 = time.perf_counter(); infer(m, xb, device); lat[i] = (time.perf_counter() - t0) * 1000
    return {"batch": batch, "device": device, "p50_ms": np.percentile(lat, 50), "p95_ms": np.percentile(lat, 95),
            "p99_ms": np.percentile(lat, 99), "mean_ms": lat.mean(),
            "us_per_patient_hour": lat.mean() * 1000 / batch, "patient_hours_per_s": batch / (lat.mean() / 1000)}

rows = []
for b in BATCH_SIZES:
    for dev in DEVICES:
        rows.append(time_device(model if dev == "cpu" else model_nnpa, dev, b))
        print(f"batch {b:>5} | {dev:>4} | p50 {rows[-1]['p50_ms']:.3f} ms | p95 {rows[-1]['p95_ms']:.3f} ms")
bench = pd.DataFrame(rows)
bench.to_csv(f"{MODEL_DIR}/bench_mlp_cpu_vs_nnpa.csv", index=False)

if NNPA_OK:
    piv = bench.pivot(index="batch", columns="device", values=["p50_ms", "p95_ms"])
    table = pd.DataFrame({"CPU p50 (ms)": piv[("p50_ms", "cpu")], "NNPA p50 (ms)": piv[("p50_ms", "nnpa")],
                          "CPU p95 (ms)": piv[("p95_ms", "cpu")], "NNPA p95 (ms)": piv[("p95_ms", "nnpa")]})
    table["speedup p50"] = table["CPU p50 (ms)"] / table["NNPA p50 (ms)"]
    display(table.style.format("{:.3f}").format({"speedup p50": "{:.1f}x"}))

In [ ]:
# Chart for the deck, README and evidence 12
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for dev, color, label in (("cpu", "#6b7280", "CPU (IFL)"), ("nnpa", "#0f62fe", "NNPA (Telum II)")):
    d = bench[bench.device == dev]
    if len(d):
        ax[0].plot(d.batch, d.p50_ms, "o-", color=color, label=label)
        ax[1].plot(d.batch, d.patient_hours_per_s, "o-", color=color, label=label)
for a, t, yl in ((ax[0], "p50 latency per batch", "ms"), (ax[1], "Throughput", "patient-hours / s")):
    a.set_xscale("log"); a.set_yscale("log"); a.set_title(t); a.set_xlabel("patients per batch"); a.set_ylabel(yl)
    a.grid(alpha=.3); a.legend()
fig.suptitle(f"SepsisWatch Z · MLP {HIDDEN} ({n_params:,} parameters) · hospital B · IBM z17")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/latency_cpu_vs_nnpa.png", dpi=150)
plt.show()
print("Figure:", f"{FIG_DIR}/latency_cpu_vs_nnpa.png")

## 9. Pitch summary

In [ ]:
t = quality[quality.split == "test"].set_index("device")
print(f"Model: MLP {HIDDEN}, {n_params:,} parameters, trained on hospital A and tested on hospital B.")
print(f"Hospital B (CPU):  AUROC {t.loc['cpu','AUROC']:.3f} | AUPRC {t.loc['cpu','AUPRC']:.3f} (prevalence {t.loc['cpu','prevalence']:.3f})")
if NNPA_OK:
    print(f"Hospital B (NNPA): AUROC {t.loc['nnpa','AUROC']:.3f} | AUPRC {t.loc['nnpa','AUPRC']:.3f}  → same quality")
    r = bench.set_index(["batch", "device"])
    b_max = max(BATCH_SIZES)
    if 12 in BATCH_SIZES:
        print(f"One ward (12 beds): CPU {r.loc[(12,'cpu'),'p50_ms']:.3f} ms vs NNPA {r.loc[(12,'nnpa'),'p50_ms']:.3f} ms")
    print(f"{b_max:,} patients: CPU {r.loc[(b_max,'cpu'),'p50_ms']:.2f} ms vs NNPA {r.loc[(b_max,'nnpa'),'p50_ms']:.2f} ms "
          f"({r.loc[(b_max,'cpu'),'p50_ms']/r.loc[(b_max,'nnpa'),'p50_ms']:.1f}x)")
    print(f"Peak NNPA throughput: {bench[bench.device=='nnpa'].patient_hours_per_s.max():,.0f} patient-hours/s")